In [1]:
!pip install kafka-python boto3 pyarrow pandas

In [ ]:
#!/usr/bin/env python3
import json, time, random, boto3, pyarrow.parquet as pq
import pyarrow.fs as fs
from kafka import KafkaProducer
from datetime import datetime
import re

# ========== CONFIGURATION ==========
KAFKA_BROKER   = "kafka:9092"
TOPIC_GPS      = "raw.gps"
MINIO_ENDPOINT = "http://100.104.4.32:9000"
MINIO_KEY      = "minioadmin"
MINIO_SECRET   = "minioadmin123"
BUCKET         = "simulated"
PREFIX         = "casa-trajectories/"
SPEED_FACTOR   = 10
GPS_INTERVAL   = 15
NOISE_SIGMA    = 0.0002
BLACKOUT_PROB  = 0.05

# ========== KAFKA PRODUCER ==========
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: str(k).encode("utf-8"),
    acks="all"
)
print("✅ Kafka connecté")

# ========== CONNEXION MINIO ==========
s3 = boto3.client("s3", endpoint_url=MINIO_ENDPOINT,
                  aws_access_key_id=MINIO_KEY, aws_secret_access_key=MINIO_SECRET)

# Lister tous les fichiers Parquet (récursif)
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
parquet_files = []
for obj in response.get("Contents", []):
    key = obj["Key"]
    if key.endswith(".parquet"):
        parquet_files.append(key)

print(f"📁 {len(parquet_files)} fichiers Parquet trouvés dans {BUCKET}/{PREFIX}")
for f in parquet_files[:5]:
    print(f"   - {f}")

if len(parquet_files) == 0:
    print("❌ Aucun fichier Parquet trouvé")
    exit(1)

arrow_fs = fs.S3FileSystem(
    endpoint_override=MINIO_ENDPOINT,
    access_key=MINIO_KEY, secret_key=MINIO_SECRET, scheme="http"
)

# ========== GÉNÉRATEUR DE TRAJETS ==========
def iter_trajectories():
    for file_key in parquet_files:
        full_path = f"{BUCKET}/{file_key}"
        print(f"📖 Lecture de {file_key}")
        table = pq.read_table(full_path, filesystem=arrow_fs)
        cols = table.column_names
        needed = ["casa_origin_lon", "casa_origin_lat", "casa_dest_lon", "casa_dest_lat",
                  "TAXI_ID", "trip_duration_min"]
        
        # Vérifier les colonnes
        missing = [col for col in needed if col not in cols]
        if missing:
            print(f"⚠️ Colonnes manquantes {missing} dans {file_key}, skip")
            continue
        
        # Extraire zone_id du chemin (ex: final_100000/origin_zone_id=5/...)
        zone_match = re.search(r"origin_zone_id[=:](\d+)", file_key)
        zone_id = int(zone_match.group(1)) if zone_match else 0
        
        data = {col: table.column(col).to_pylist() for col in needed}
        for i in range(table.num_rows):
            yield {
                "TAXI_ID": data["TAXI_ID"][i],
                "casa_origin_lon": data["casa_origin_lon"][i],
                "casa_origin_lat": data["casa_origin_lat"][i],
                "casa_dest_lon": data["casa_dest_lon"][i],
                "casa_dest_lat": data["casa_dest_lat"][i],
                "trip_duration_min": data["trip_duration_min"][i],
                "origin_zone_id": zone_id
            }

def add_noise(lat, lon):
    return lat + random.gauss(0, NOISE_SIGMA), lon + random.gauss(0, NOISE_SIGMA)

def simulate_trip(row):
    taxi_id = int(row["TAXI_ID"])
    o_lon, o_lat = row["casa_origin_lon"], row["casa_origin_lat"]
    d_lon, d_lat = row["casa_dest_lon"], row["casa_dest_lat"]
    duration_min = float(row["trip_duration_min"])
    zone_id = row["origin_zone_id"]

    n_points = max(4, int(duration_min * 4))

    for i in range(n_points):
        if random.random() < BLACKOUT_PROB:
            delay = random.uniform(60, 180)
            time.sleep(delay / SPEED_FACTOR)
            continue

        ratio = i / (n_points - 1) if n_points > 1 else 0
        lon = o_lon + (d_lon - o_lon) * ratio
        lat = o_lat + (d_lat - o_lat) * ratio
        lat_noisy, lon_noisy = add_noise(lat, lon)

        event = {
            "taxi_id": taxi_id,
            "timestamp": int(datetime.now().timestamp()),
            "lat": round(lat_noisy, 6),
            "lon": round(lon_noisy, 6),
            "speed": round(random.uniform(10, 70), 1),
            "status": "available" if i == 0 else "on_trip",
            "zone_id": zone_id
        }
        producer.send(TOPIC_GPS, key=str(taxi_id), value=event)
        time.sleep(GPS_INTERVAL / SPEED_FACTOR)

    producer.send(TOPIC_GPS, key=str(taxi_id), value={
        "taxi_id": taxi_id,
        "timestamp": int(datetime.now().timestamp()),
        "lat": round(add_noise(d_lat, d_lon)[0], 6),
        "lon": round(add_noise(d_lat, d_lon)[1], 6),
        "speed": 0.0,
        "status": "available",
        "zone_id": zone_id
    })

print("🚕 GPS Producer démarré")
print(f"   Topic : {TOPIC_GPS} | Vitesse x{SPEED_FACTOR}")
count = 0
try:
    while True:
        for row in iter_trajectories():
            simulate_trip(row)
            count += 1
            if count % 20 == 0:
                producer.flush()
                print(f"   ✅ {count} trajets GPS envoyés")
        print("🔄 Replay complet des données...")
except KeyboardInterrupt:
    print(f"\n⏹️ Arrêt — {count} trajets envoyés")
    producer.flush()
    producer.close()

In [6]:
from kafka import KafkaProducer
import json

try:
    producer = KafkaProducer(
        bootstrap_servers="kafka:9092",
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
        max_block_ms=5000
    )
    print("✅ Connecté à kafka:9092 depuis Jupyter")
    producer.close()
except Exception as e:
    print(f"❌ Erreur: {e}")

✅ Connecté à kafka:9092 depuis Jupyter


In [10]:
import subprocess
result = subprocess.run(['cat', '/etc/hosts'], capture_output=True, text=True)
print(result.stdout)

127.0.0.1	localhost
::1	localhost ip6-localhost ip6-loopback
fe00::	ip6-localnet
ff00::	ip6-mcastprefix
ff02::1	ip6-allnodes
ff02::2	ip6-allrouters
172.18.0.11	07ddb89c11f9



In [18]:
#!/usr/bin/env python3
import json, time, random, boto3, io, pyarrow.parquet as pq
from kafka import KafkaProducer
from datetime import datetime
import re

# ========== CONFIGURATION ==========
KAFKA_BROKER   = "172.18.0.2:9092"     # IP directe de Kafka
TOPIC_GPS      = "raw.gps"
MINIO_ENDPOINT = "http://100.104.4.32:9000"
MINIO_KEY      = "minioadmin"
MINIO_SECRET   = "minioadmin123"
BUCKET         = "simulated"
PREFIX         = "casa-trajectories/final_89688/"
SPEED_FACTOR   = 10
GPS_INTERVAL   = 0.5                    # 0.5 seconde entre les messages
NOISE_SIGMA    = 0.0002
BLACKOUT_PROB  = 0.05

# ========== KAFKA PRODUCER ==========
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: str(k).encode("utf-8"),
    acks=1,
    retries=3,
    max_block_ms=60000,
    request_timeout_ms=30000,
    api_version=(3, 4, 0)
)
print("✅ Kafka connecté")

# ========== CONNEXION MINIO ==========
s3 = boto3.client("s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_KEY,
    aws_secret_access_key=MINIO_SECRET,
    config=boto3.session.Config(signature_version='s3v4')
)

# Lister les fichiers Parquet
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
parquet_files = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".parquet")]
print(f"📁 {len(parquet_files)} fichiers Parquet trouvés")
for f in parquet_files:
    print(f"   - {f}")

def read_parquet_via_boto3(bucket, key):
    """Lit un fichier Parquet depuis MinIO via boto3"""
    response = s3.get_object(Bucket=bucket, Key=key)
    return pq.read_table(io.BytesIO(response['Body'].read()))

def extract_coords_from_polyline(polyline_str):
    """Extrait le premier et dernier point d'une polyline JSON"""
    try:
        coords = json.loads(polyline_str)
        if coords and len(coords) > 0:
            return coords[0][0], coords[0][1], coords[-1][0], coords[-1][1]
    except:
        pass
    return None, None, None, None

def add_noise(lat, lon):
    """Ajoute du bruit GPS"""
    return lat + random.gauss(0, NOISE_SIGMA), lon + random.gauss(0, NOISE_SIGMA)

print("🚕 GPS Producer démarré")
print(f"   Topic : {TOPIC_GPS}")
count = 0

try:
    for file_key in parquet_files:
        print(f"\n📖 Lecture de {file_key}")
        table = read_parquet_via_boto3(BUCKET, file_key)
        
        cols = table.column_names
        print(f"   Colonnes: {cols}")
        
        # Récupérer les données
        taxi_ids = table.column("TAXI_ID").to_pylist()
        durations = table.column("DURATION_S").to_pylist()
        polylines = table.column("POLYLINE_CASA").to_pylist()
        
        # Extraire zone_id du chemin (ex: final_89688/origin_zone_id=5/...)
        zone_match = re.search(r"origin_zone_id[=:](\d+)", file_key)
        zone_id = int(zone_match.group(1)) if zone_match else 1
        
        # Parcourir tous les trajets
        for i in range(table.num_rows):
            start_lon, start_lat, end_lon, end_lat = extract_coords_from_polyline(polylines[i])
            if start_lon is None:
                continue
            
            taxi_id = int(taxi_ids[i])
            duration_min = float(durations[i]) / 60.0
            n_points = max(2, int(duration_min * 4))  # Nombre de points GPS
            
            # Simuler le trajet avec plusieurs points GPS
            for j in range(n_points):
                if random.random() < BLACKOUT_PROB:
                    continue
                
                ratio = j / (n_points - 1) if n_points > 1 else 0
                lon = start_lon + (end_lon - start_lon) * ratio
                lat = start_lat + (end_lat - start_lat) * ratio
                lat_noisy, lon_noisy = add_noise(lat, lon)
                
                event = {
                    "taxi_id": taxi_id,
                    "timestamp": int(datetime.now().timestamp()),
                    "lat": round(lat_noisy, 6),
                    "lon": round(lon_noisy, 6),
                    "speed": round(random.uniform(10, 70), 1),
                    "status": "available" if j == 0 else "on_trip",
                    "zone_id": zone_id
                }
                
                # Envoyer au topic Kafka
                future = producer.send(TOPIC_GPS, key=str(taxi_id), value=event)
                future.get(timeout=10)  # Attendre confirmation
                count += 1
                
                if count % 10 == 0:
                    print(f"   ✅ {count} messages envoyés (taxi {taxi_id})")
                
                time.sleep(GPS_INTERVAL / SPEED_FACTOR)
            
            # Dernier message (arrivée)
            event = {
                "taxi_id": taxi_id,
                "timestamp": int(datetime.now().timestamp()),
                "lat": round(add_noise(end_lat, end_lon)[0], 6),
                "lon": round(add_noise(end_lat, end_lon)[1], 6),
                "speed": 0.0,
                "status": "available",
                "zone_id": zone_id
            }
            producer.send(TOPIC_GPS, key=str(taxi_id), value=event)
            count += 1
            
            if count % 10 == 0:
                print(f"   ✅ {count} messages envoyés (arrivée taxi {taxi_id})")
            
            # === PAS DE LIMITE ===
            # Le script continue jusqu'à épuisement des trajets
            # Pour arrêter manuellement : Ctrl+C dans Jupyter ou PowerShell
                
except KeyboardInterrupt:
    print(f"\n⏹️ Arrêt manuel — {count} messages envoyés")
except Exception as e:
    print(f"❌ Erreur: {e}")
finally:
    producer.flush()
    producer.close()
    print(f"✅ Terminé — {count} messages envoyés au total")

✅ Kafka connecté
📁 2 fichiers Parquet trouvés
   - casa-trajectories/final_89688/part-00000-14480b7d-931b-441c-84fe-8a15cadedc10-c000.snappy.parquet
   - casa-trajectories/final_89688/part-00001-14480b7d-931b-441c-84fe-8a15cadedc10-c000.snappy.parquet
🚕 GPS Producer démarré
   Topic : raw.gps

📖 Lecture de casa-trajectories/final_89688/part-00000-14480b7d-931b-441c-84fe-8a15cadedc10-c000.snappy.parquet
❌ Erreur: Read timeout on endpoint URL: "None"
✅ Terminé — 0 messages envoyés au total


In [ ]:
# ========== BOUCLE PRINCIPALE ==========
print("🚕 GPS Producer démarré")
print(f"   Topic : {TOPIC_GPS} | Vitesse x{SPEED_FACTOR}")
count = 0
try:
    while True:
        for row in iter_trajectories():
            simulate_trip(row)
            count += 1
            if count % 20 == 0:
                producer.flush()
                print(f"   ✅ {count} trajets GPS envoyés")
        print("🔄 Replay complet des données...")
except KeyboardInterrupt:
    print(f"\n⏹️ Arrêt — {count} trajets envoyés")
    producer.flush()
    producer.close()

In [ ]:
gps.terminate()